# Deep Learning-Based Anomaly Detection and Latent Representation Analysis

A self-contained notebook with PyTorch autoencoder training, synthetic anomaly detection, model evaluation, and visualizations.

**Note:** This is a synthetic-data demonstration, not a validated real-world anomaly detector.

In [ ]:
# Complete project in one code cell
import random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import joblib
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_curve, auc, confusion_matrix, classification_report, precision_recall_curve, average_precision_score, accuracy_score, precision_score, recall_score, f1_score)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid', context='notebook')
print('DEEP LEARNING-BASED ANOMALY DETECTION | Device:', DEVICE)

# 1. Synthetic data
N_NORMAL, N_ANOMALIES, N_FEATURES = 4000, 400, 8
X_normal, _ = make_blobs(n_samples=N_NORMAL, centers=4, n_features=N_FEATURES, cluster_std=1.25, random_state=SEED)
rng = np.random.default_rng(SEED + 1)
X_anomaly = rng.uniform(low=-10, high=10, size=(N_ANOMALIES, N_FEATURES))
X = np.vstack([X_normal, X_anomaly])
y = np.concatenate([np.zeros(N_NORMAL, dtype=int), np.ones(N_ANOMALIES, dtype=int)])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
X_train_normal_raw = X_train_raw[y_train == 0]
scaler = StandardScaler().fit(X_train_normal_raw)
X_train_normal = scaler.transform(X_train_normal_raw)
X_test = scaler.transform(X_test_raw)
train_tensor = torch.tensor(X_train_normal, dtype=torch.float32)
test_tensor = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=128, shuffle=True)
print(f'Samples: {len(X)} | Normal: {(y == 0).sum()} | Anomalies: {(y == 1).sum()}')

# 2. Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim=8, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 16), nn.ReLU(), nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, input_dim))
    def encode(self, x): return self.encoder(x)
    def forward(self, x): return self.decoder(self.encode(x))

model = Autoencoder(input_dim=N_FEATURES, latent_dim=2).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
print(model)

# 3. Training
EPOCHS = 100
loss_history = []
for epoch in range(EPOCHS):
    model.train(); total_loss = 0.0; total_samples = 0
    for (batch,) in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        reconstruction = model(batch)
        loss = criterion(reconstruction, batch)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(batch); total_samples += len(batch)
    epoch_loss = total_loss / total_samples
    loss_history.append(epoch_loss)
    if (epoch + 1) % 20 == 0: print(f'Epoch {epoch+1:03d}/{EPOCHS} | Loss: {epoch_loss:.6f}')

# 4. Scores and latent vectors
model.eval()
with torch.no_grad():
    reconstructed_test = model(test_tensor)
    anomaly_scores = torch.mean((test_tensor - reconstructed_test) ** 2, dim=1).cpu().numpy()
    latent = model.encode(test_tensor).cpu().numpy()
    normal_tensor = torch.tensor(X_train_normal, dtype=torch.float32, device=DEVICE)
    reconstructed_normal = model(normal_tensor)
    normal_train_scores = torch.mean((normal_tensor - reconstructed_normal) ** 2, dim=1).cpu().numpy()
threshold = float(np.percentile(normal_train_scores, 95))
predictions = (anomaly_scores > threshold).astype(int)

# 5. Metrics
fpr, tpr, _ = roc_curve(y_test, anomaly_scores)
roc_auc = auc(fpr, tpr)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, anomaly_scores)
average_precision = average_precision_score(y_test, anomaly_scores)
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
accuracy = accuracy_score(y_test, predictions)
precision_value = precision_score(y_test, predictions, zero_division=0)
recall_value = recall_score(y_test, predictions, zero_division=0)
f1_value = f1_score(y_test, predictions, zero_division=0)
print(f'ROC-AUC: {roc_auc:.4f} | Average Precision: {average_precision:.4f}')
print(f'Accuracy: {accuracy:.4f} | Precision: {precision_value:.4f} | Recall: {recall_value:.4f} | F1: {f1_value:.4f}')
print(classification_report(y_test, predictions, target_names=['Normal', 'Anomaly'], zero_division=0))

# 6. Six-panel dashboard
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes[0, 0].plot(range(1, EPOCHS + 1), loss_history, linewidth=2)
axes[0, 0].set(title='1. Autoencoder Training Loss', xlabel='Epoch', ylabel='Mean Squared Error')
axes[0, 1].hist(anomaly_scores[y_test == 0], bins=35, alpha=0.7, label='Normal')
axes[0, 1].hist(anomaly_scores[y_test == 1], bins=35, alpha=0.7, label='Anomaly')
axes[0, 1].axvline(threshold, linestyle='--', linewidth=2, label='Training-derived threshold')
axes[0, 1].set(title='2. Anomaly Score Distribution', xlabel='Reconstruction Error', ylabel='Frequency'); axes[0, 1].legend()
sc = axes[0, 2].scatter(latent[:, 0], latent[:, 1], c=y_test, cmap='coolwarm', alpha=0.8, s=22)
axes[0, 2].set(title='3. Learned Latent Space', xlabel='Latent Dimension 1', ylabel='Latent Dimension 2'); fig.colorbar(sc, ax=axes[0, 2], label='Actual class')
sc2 = axes[1, 0].scatter(latent[:, 0], latent[:, 1], c=anomaly_scores, cmap='viridis', alpha=0.8, s=22)
axes[1, 0].set(title='4. Latent Space Colored by Anomaly Score', xlabel='Latent Dimension 1', ylabel='Latent Dimension 2'); fig.colorbar(sc2, ax=axes[1, 0], label='Reconstruction error')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'], ax=axes[1, 1])
axes[1, 1].set(title='5. Confusion Matrix', xlabel='Predicted label', ylabel='Actual label')
axes[1, 2].plot(fpr, tpr, linewidth=2, label=f'ROC-AUC = {roc_auc:.3f}'); axes[1, 2].plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
axes[1, 2].set(title='6. ROC Curve', xlabel='False Positive Rate', ylabel='True Positive Rate'); axes[1, 2].legend()
fig.suptitle('Deep Learning-Based Anomaly Detection Analysis', fontsize=18, fontweight='bold')
plt.tight_layout(); plt.show()

# 7. Precision-recall curve
plt.figure(figsize=(9, 6)); plt.plot(recall_curve, precision_curve, linewidth=2, label=f'Average Precision = {average_precision:.3f}')
plt.title('7. Precision-Recall Curve'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.tight_layout(); plt.show()

# 8. PCA visualization
pca = PCA(n_components=2, random_state=SEED)
X_test_pca = pca.fit_transform(X_test)
plt.figure(figsize=(10, 7))
for label, name in [(0, 'Normal'), (1, 'Anomaly')]:
    mask = y_test == label
    plt.scatter(X_test_pca[mask, 0], X_test_pca[mask, 1], alpha=0.7, s=25, label=name)
plt.title('8. PCA Projection of Test Data'); plt.xlabel('Principal Component 1'); plt.ylabel('Principal Component 2'); plt.legend(); plt.tight_layout(); plt.show()
print(f'Variance explained by first two PCA components: {pca.explained_variance_ratio_.sum():.2%}')

# 9. Original vs reconstructed feature values
normal_indices = np.where(y_test == 0)[0]
anomaly_indices = np.where(y_test == 1)[0]
selected_indices = np.concatenate([normal_indices[:3], anomaly_indices[:3]])
with torch.no_grad(): selected_reconstructed = model(test_tensor[selected_indices]).cpu().numpy()
selected_original = X_test[selected_indices]
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for i, idx in enumerate(selected_indices):
    kind = 'Anomaly' if y_test[idx] == 1 else 'Normal'
    axes[0].plot(selected_original[i], marker='o', label=f'Sample {i+1} ({kind})')
    axes[1].plot(selected_reconstructed[i], marker='o', label=f'Sample {i+1} ({kind})')
axes[0].set(title='9. Original Standardized Feature Values', xlabel='Feature Index', ylabel='Value'); axes[0].legend(ncol=3)
axes[1].set(title='10. Autoencoder-Reconstructed Feature Values', xlabel='Feature Index', ylabel='Reconstructed value'); axes[1].legend(ncol=3)
plt.tight_layout(); plt.show()

# 10. Export files
results_df = pd.DataFrame({'Actual_Label': y_test, 'Anomaly_Score': anomaly_scores, 'Predicted_Anomaly': predictions, 'Threshold': threshold, 'Latent_Dimension_1': latent[:, 0], 'Latent_Dimension_2': latent[:, 1]})
results_df.to_csv('deep_learning_anomaly_results.csv', index=False)
pd.DataFrame({'Epoch': np.arange(1, EPOCHS + 1), 'Training_Loss': loss_history}).to_csv('autoencoder_training_history.csv', index=False)
torch.save(model.state_dict(), 'deep_autoencoder_weights.pth')
joblib.dump(scaler, 'anomaly_detection_scaler.joblib')
print('EXPERIMENT COMPLETED. Output files saved in the current working directory.')
